In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from dataset import FER2013Dataset   # from your file :contentReference[oaicite:0]{index=0}
from model import EmotionResNet      # from your file :contentReference[oaicite:1]{index=1}

import numpy as np
from tqdm import tqdm

In [2]:
CONFIG = {
    "csv": "fer2013.csv",
    "batch_size": 128,
    "epochs": 20,   # reduce for testing
    "lr": 1e-3,
    "workers": 2
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
def build_transforms():
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.Resize(56),
        transforms.RandomCrop(48),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
        transforms.RandomErasing(p=0.3)
    ])

    val_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std)
    ])

    return train_tf, val_tf

train_tf, val_tf = build_transforms()

In [5]:
train_ds = FER2013Dataset(CONFIG["csv"], "Training", train_tf)
val_ds = FER2013Dataset(CONFIG["csv"], "PublicTest", val_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["workers"],
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"] * 2,
    shuffle=False,
    num_workers=CONFIG["workers"],
    pin_memory=True
)

print("Train:", len(train_ds), "Val:", len(val_ds))

Train: 28709 Val: 3589


In [6]:
counts = train_ds.label_counts()
weights = counts.sum() / (len(counts) * counts.astype(np.float32))
weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("Class weights:", weights)

Class weights: tensor([1.0266, 9.4066, 1.0010, 0.5684, 0.8491, 1.2934, 0.8260],
       device='cuda:0')


In [14]:
model = EmotionResNet(num_classes=7).to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["epochs"]
)
scaler = torch.cuda.amp.GradScaler()

In [18]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    loss_sum = 0.0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        loss_sum += loss.item() * labels.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    loss = loss_sum / total

    return acc

In [19]:
scaler = torch.cuda.amp.GradScaler()

best_acc = 0.0

for epoch in range(CONFIG["epochs"]):
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for imgs, labels in pbar:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # ✅ AMP INSIDE LOOP
        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix(loss=running_loss / (pbar.n + 1))

    scheduler.step()

    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Saved best model")

Epoch 1: 100%|██████████| 225/225 [00:15<00:00, 14.40it/s, loss=1.88]


Epoch 1 | Val Acc: 0.4831
✅ Saved best model


Epoch 2: 100%|██████████| 225/225 [00:15<00:00, 14.93it/s, loss=1.78]


Epoch 2 | Val Acc: 0.4837
✅ Saved best model


Epoch 3: 100%|██████████| 225/225 [00:15<00:00, 14.46it/s, loss=1.73]


...

Epoch 18 | Val Acc: 0.6166


Epoch 19: 100%|██████████| 225/225 [00:17<00:00, 12.64it/s, loss=1.41]


Epoch 19 | Val Acc: 0.6208
✅ Saved best model


Epoch 20: 100%|██████████| 225/225 [00:17<00:00, 12.55it/s, loss=1.41]


Epoch 20 | Val Acc: 0.6177


In [16]:
import torch
print(torch.__version__)

2.2.2+cu121


In [20]:
torch.save(model.state_dict(), "best_model_face.pth")

In [ ]:
model = EmotionResNet(num_classes=7).to(device)
model.load_state_dict(torch.load("best_model_face.pth", map_location=device))
model.eval()

EmotionResNet(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tr

: 

In [22]:
dummy_input = torch.randn(1, 3, 48, 48).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "emotion_model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    opset_version=11
)

print("✅ ONNX model saved")

✅ ONNX model saved


In [23]:
import onnx

onnx_model = onnx.load("emotion_model.onnx")
onnx.checker.check_model(onnx_model)

print("✅ ONNX model is valid")

✅ ONNX model is valid


python realtime.py --weights best_model.pth (enter in terminal forn real time )
